# Overview

# Frequency of inlet check valve opening
Based on the previous plot, we have identified a set of barrier underpressure conditions where the inlet check valve will open. Since the compensating inlet flow is assumed to be instantaneous which is not conservative, we should show how often the check valve opens over a variety of max flow rates.

To do this, we will use real ambient pressure data. 

For each possible max inlet flow rate which defines the check valve cracking threshold, we will filter the pressure data by dp/dt > this threshold, and make sure to only count underpressure events that take place over multiple time steps once.

More significantly than the frequency of opening is the total time the check valve is open over the entire data period.

Plot the fraction of the data the check valve is open vs max inlet flow rate.

## Imports

In [ ]:
# Plotting
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

# Vector math
import numpy as np

# Dataframes
import pandas as pd

# Data grouping
from scipy.ndimage import label

# matplotlib plotting settings
plt.rcParams.update({
    'font.size': 14,
    'axes.titlesize': 18,
    'axes.labelsize': 16,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    'legend.fontsize': 14
})

# Import constants
import config as cfg

# Import fluid dynamics methods
from methods import (
    calculate_cross_sectional_area,
    calculate_reynolds_number,
    calculate_friction_factor,
    calculate_darcy_weisbach_impedance,
    pressure_increase_rate
)

## Load cavern pressure time series

In [88]:
# Load the data into a dataframe
cavern_df = pd.read_csv("local_data/atmPressure-CUTE-csv.csv")
cavern_df.columns

Index(['index', 'datetime', 'pressure'], dtype='str')

## Define system and valve parameters

In [89]:
# Define parameters of system
max_flow = 80   # SLPM
volume = 4      # cubic meters
temperature = 298           # Room temp, kelvins

# Nominal baseline setpoint
setpoint_dp = 0.001 

# Valve characteristics
valve_cracking_pressure = -0.1  # psid

# 100 SLPM at -0.1 psi (cracking pressure)
opening_flow = 700
base_valve_rate = pressure_increase_rate(opening_flow, volume, temperature) / cfg.psi_to_pa

## Clean data

This involves regularizing the datetime, and interpolating missing data.

This is required since computation of flow through the opened check valve depends on the pressure difference every timestep -- missing data will cause erros when calculating differential pressure between successive times.

Select a date range for the time series that is most helpful to your analysis.

In [ ]:
# Standardize and sort data
cavern_df["datetime"] = pd.to_datetime(
    cavern_df["datetime"], format="'%Y-%m-%d %H:%M:%S'"
)
cavern_df = cavern_df.sort_values(by="datetime").drop_duplicates(subset=["datetime"])

# Clip time range to recent years, most relevant to current environment
start_date = pd.to_datetime("2024-01-01 00:00:00")  # Example start datetime
end_date = pd.to_datetime("2025-02-07 02:30:31")    # Example end datetime
cavern_df = cavern_df[(cavern_df["datetime"] >= start_date) & (cavern_df["datetime"] <= end_date)]

cavern_df["pressure"] = pd.to_numeric(cavern_df["pressure"], errors="coerce")

freq = pd.Timedelta(seconds=1)
# Identify the inherent time step of your dataset (e.g., 1 minute, 5 minutes)
# We find the most common time delta between existing consecutive rows
# freq = cavern_df["datetime"].diff().mode()[0]
print(f"frequency: {freq}")

# Create a complete, unbroken datetime grid from start to finish
full_time_grid = pd.date_range(
    start=cavern_df["datetime"].min(), end=cavern_df["datetime"].max(), freq=freq
)

# Reindex the dataframe to inject the missing rows
cavern_df_filled = (
    cavern_df.set_index("datetime")
    .reindex(full_time_grid)
    .rename_axis("datetime")
    .reset_index()
)

# Linearly interpolate the missing pressure values across the new gaps
# This prevents NaN propagation while keeping the physical transition smooth
cavern_df_filled["pressure"] = cavern_df_filled["pressure"].interpolate(method="linear")

# Convert pressure to PSI and compute derivatives safely
cavern_df_filled["pressure_psi"] = cavern_df_filled["pressure"] * 0.0145038

cavern_df_filled["dP_psi"] = cavern_df_filled["pressure_psi"].diff()
cavern_df_filled["dt_min"] = cavern_df_filled["datetime"].diff().dt.total_seconds() / 60
cavern_df_filled["dp_dt_psi_per_min"] = cavern_df_filled["dP_psi"] / cavern_df_filled["dt_min"]

# Drop only the very first row (which is a natural NaN from the diff)
cavern_df_clean = cavern_df_filled.dropna(
    subset=["dp_dt_psi_per_min", "dt_min"]
).reset_index(drop=True)

# Verify time range of data
realstart = cavern_df_clean["datetime"].min()
realend = cavern_df_clean["datetime"].max()
print(f"{realstart} - {realend}")
print(f"{pd.to_timedelta(realend - realstart)}")

cavern_df_clean.head(10)

frequency: 0 days 00:00:01
2024-01-01 00:00:12 - 2025-02-07 02:30:31
403 days 02:30:19


,datetime,index,pressure,pressure_psi,dP_psi,dt_min,dp_dt_psi_per_min
0,2024-01-01 00:00:13,NaN,1248.44,18.107124,0.0,0.016667,0.0
1,2024-01-01 00:00:14,NaN,1248.44,18.107124,0.0,0.016667,0.0
2,2024-01-01 00:00:15,NaN,1248.44,18.107124,0.0,0.016667,0.0
3,2024-01-01 00:00:16,NaN,1248.44,18.107124,0.0,0.016667,0.0
4,2024-01-01 00:00:17,NaN,1248.44,18.107124,0.0,0.016667,0.0
5,2024-01-01 00:00:18,NaN,1248.44,18.107124,0.0,0.016667,0.0
6,2024-01-01 00:00:19,NaN,1248.44,18.107124,0.0,0.016667,0.0
7,2024-01-01 00:00:20,NaN,1248.44,18.107124,0.0,0.016667,0.0
8,2024-01-01 00:00:21,NaN,1248.44,18.107124,0.0,0.016667,0.0
9,2024-01-01 00:00:22,NaN,1248.44,18.107124,0.0,0.016667,0.0


In [91]:
# Define max inlet psi/min rates
undepressure_df = pd.DataFrame()
undepressure_df["Max flow rates (SLPM)"] = np.linspace(10, max_flow, 20)
undepressure_df["Max DP rates (PSI/min)"] = \
    undepressure_df.apply(
        lambda row: pressure_increase_rate(
            row["Max flow rates (SLPM)"], 
            volume, 
            temperature
        ) / cfg.psi_to_pa,
        axis=1
    )

## Analysis loop

This loop assesses the amount of time and frequency of inlet check valve cracking over the prssure time series.

It gives an upper and lower bound, considering only the flow controller and both flow controller and check valve flow (neglecting the valves impedance).

The vlave flow at each time interval depends on the differential pressure. We optimistically assume that it has a direct proportionality with the pressure excess above its opening setpoint. But proper calculation requires knowing the flow coefficient and connecting tubing impedance.

In [78]:
def run_underpressure_simulation(
    dt_array,
    ambient_rate_array,
    initial_dp,
    valve_cracking_pressure,
    valve_opening_rate,
    control_rate
):

    barrier_dp = setpoint_dp  # Reset internal volume pressure state
    total_open_time = 0.0
    valve_status_history = np.zeros(len(dt_array), dtype=bool)

    for i in range(len(dt_array)):
        dt = dt_array[i]
        ambient_rate = ambient_rate_array[i]
        
        # Check if we are past the cracking pressure
        if barrier_dp <= valve_cracking_pressure:
            # Delta past cracking pressure (e.g., if barrier is -0.12, delta is 0.02)
            delta_past_cracking = valve_cracking_pressure - barrier_dp
            
            # Flow rate is directly proportional to how far past cracking we are
            current_valve_rate = valve_opening_rate * (1 + delta_past_cracking)
            
            # Valve is open
            valve_status_history[i] = True
            total_open_time += dt
        else:
            current_valve_rate = 0.0
            
        # Update the state of the barrier volume pressure
        # (Internal goes up due to controller + valve, down due to ambient rise)
        barrier_dp += (control_rate + current_valve_rate - ambient_rate) * dt
        
        # Upper clamp: Controller throttles down when it recovers back to setpoint
        if barrier_dp > setpoint_dp:
            barrier_dp = setpoint_dp
            
    return valve_status_history, total_open_time

In [92]:
# Extract arrays from dataframes for speed
dt_array = cavern_df_clean["dt_min"].to_numpy()
dp_dt_array = cavern_df_clean["dp_dt_psi_per_min"].to_numpy()
total_time = dt_array.sum()     # minutes

In [94]:
# Prepare lists to store the final curves for plotting
fraction_conservative = []
distinct_events_cons = []
fraction_optimistic = []
distinct_events_opt = []

# Loop through each candidate controller capacity
for _, row in undepressure_df.iterrows():
    ctrl_rate = row["Max DP rates (PSI/min)"]
    
    # ----------------------------------------------------
    # 1. CONSERVATIVE BOUND (Pure Rate Filter)
    # ----------------------------------------------------
    is_overwhelmed = dp_dt_array > ctrl_rate
    time_open_cons = dt_array[is_overwhelmed].sum()
    fraction_conservative.append(time_open_cons / total_time)

    # Count Distinct Events
    # An event starts when it is overwhelmed now, but wasn't in the previous step
    event_starts_cons = is_overwhelmed & (~np.roll(is_overwhelmed, 1))

    # Negate the roll artifact of wrapping last element to index 0
    if len(is_overwhelmed) > 0:
        event_starts_cons[0] = is_overwhelmed[0]
    distinct_events_cons.append(event_starts_cons.sum())

    # ----------------------------------------------------
    # 2. OPTIMISTIC BOUND (Proportional Flow Simulation)
    # ----------------------------------------------------
    valve_status_history, valve_open_time = run_underpressure_simulation(
            dt_array,
            dp_dt_array,
            setpoint_dp,
            valve_cracking_pressure,
            base_valve_rate,
            ctrl_rate
        )

    fraction_optimistic.append(valve_open_time / total_time)
    
    # Count distinct simulated valve events
    event_starts = valve_status_history & (~np.roll(valve_status_history, 1))
    if len(valve_status_history) > 0:
        event_starts[0] = valve_status_history[0] 
    distinct_events_opt.append(event_starts.sum())

# Convert final lists to arrays for your Plotly graphing functions
fraction_time_open_conservative = np.array(fraction_conservative)
event_counts_cons = np.array(distinct_events_cons)
fraction_time_open_optimistic = np.array(fraction_optimistic)
event_counts_opt = np.array(distinct_events_opt)

In [95]:
# Convert to more meaningful quantities for plotting
# Convert total number of opening events to per year (use minutes per year)
event_counts_cons_per_year = (event_counts_cons / total_time) * 525600
event_counts_opt_per_year = (event_counts_opt / total_time) * 525600

In [111]:
# X-axis sweep array
x_flow_rates = undepressure_df["Max flow rates (SLPM)"].to_numpy()

assumptions_str = f"<br><sup>Ignores intake through exhaust | Assumes instant inlet flow | Assumes {opening_flow} SLPM valve flow at 0.1 psid scales linearly with differential pressure</sup>"
upper_bound_str = "Upper Bound (No valve flow)"
lower_bound_str = "Lower bound (With optimistic valve flow)"

# ----------------------------------------------------
# PLOT 1: TOTAL EXPOSURE TIME (%)
# ----------------------------------------------------
fig_time = go.Figure()

# Upper Bound - Conservative
fig_time.add_trace(
    go.Scatter(
        x=x_flow_rates,
        y=fraction_time_open_conservative * 100,
        mode="lines",
        line=dict(color="rgba(220, 53, 69, 0.4)", width=2, dash="dash"),
        name=upper_bound_str,
        hovertemplate="<b>Max Flow:</b> %{x:.1f} SLPM<br><b>Upper Limit:</b> %{y:.4f}%<extra></extra>",
    )
)

# Lower Bound - Optimistic (with Area Fill)
fig_time.add_trace(
    go.Scatter(
        x=x_flow_rates,
        y=fraction_time_open_optimistic * 100,
        mode="lines",
        line=dict(color="rgba(0, 123, 255, 0.8)", width=3),
        name=lower_bound_str,
        fill="tonexty",
        fillcolor="rgba(0, 123, 255, 0.1)",
        hovertemplate="<b>Max Flow:</b> %{x:.1f} SLPM<br><b>Lower Limit:</b> %{y:.4f}%<extra></extra>",
    )
)

fig_time.update_yaxes(type="log")
fig_time.update_layout(
    title=dict(
        text=(
            "<b>Check Valve Total Exposure Time vs. Inlet Flow Capacity</b>"
            "<br><sup>The calculated range of total valve open duration based on ~2 years of recent cavern pressure time series data.</sup>"
            f"{assumptions_str}"
        ),
        font=dict(size=14),
        x=0.5,
        xanchor="center",
        y=0.93,
    ),
    xaxis=dict(title="Maximum Inlet Flow Rate Capacity (SLPM)", gridcolor="rgba(200, 200, 200, 0.15)"),
    yaxis=dict(title="Total Fraction of Dataset Valve is Open (%)", gridcolor="rgba(200, 200, 200, 0.15)", zeroline=False),
    template="plotly_white",
    margin=dict(t=100, b=60, l=60, r=60),
    width=900,
    height=480,
    hovermode="x unified",
    legend=dict(yanchor="top", y=0.95, xanchor="right", x=0.98)
)


# ----------------------------------------------------
# PLOT 2: DISTINCT ACTUATION EVENTS (Counts)
# ----------------------------------------------------
fig_events = go.Figure()

# Upper bound Event Counts
fig_events.add_trace(
    go.Scatter(
        x=x_flow_rates,
        y=event_counts_cons_per_year,
        mode="lines+markers",
        name=upper_bound_str,
        line=dict(color="#d62728", width=2, dash="dash"),
        marker=dict(symbol="x", size=6),
        hovertemplate="<b>Max Flow:</b> %{x:.1f} SLPM<br><b>Cons. Events per year:</b> %{y:.2f}<extra></extra>",
    )
)

# Lower bound Event Counts
fig_events.add_trace(
    go.Scatter(
        x=x_flow_rates,
        y=event_counts_opt_per_year,
        mode="lines+markers",
        name=lower_bound_str,
        line=dict(color="#2ca02c", width=2.5),
        marker=dict(symbol="circle", size=6),
        fill="tonexty",
        fillcolor="rgba(0, 123, 255, 0.1)",
        hovertemplate="<b>Max Flow:</b> %{x:.1f} SLPM<br><b>Sim. Events per year:</b> %{y:.2f}<extra></extra>",
    )
)

fig_events.update_yaxes(type="log")
fig_events.update_layout(
    title=dict(
        text=(
            "<b>Valve Actuation Events vs. Inlet Flow Capacity</b>"
            "<br><sup>How often the intake valve opens based on ~2 years of recent cavern pressure time series data.</sup>"
            f"<br><sup>Ignores intake through exhaust | Assumes instant inlet flow | Assumes {opening_flow} SLPM valve flow at 0.1 psid and linear scaling with differential pressure</sup>"
        ),
        font=dict(size=14),
        x=0.5,
        xanchor="center",
        y=0.93,
    ),
    xaxis=dict(title="Maximum Inlet Flow Rate Capacity (SLPM)", gridcolor="rgba(200, 200, 200, 0.15)"),
    yaxis=dict(title="Number of Opening Events per Year", gridcolor="rgba(200, 200, 200, 0.15)", zeroline=False),
    template="plotly_white",
    margin=dict(t=100, b=60, l=60, r=60),
    width=900,
    height=480,
    hovermode="x unified",
    legend=dict(yanchor="top", y=0.95, xanchor="right", x=0.98)
)

# Save
fig_time.write_html("underpressure_time_vs_inlet_flows.html")
fig_events.write_html("underpressure_freq_vs_inlet_flows.html")

# Render both separately
fig_time.show()
fig_events.show()

## Opening events vs event duration 

The above plots are not meaningful by themselves because the valve could open many times but only for short periods, accumulating a large amount of total time, but not having a signifincant effect on the concentration of cavern air in the volume.

In [99]:
# --- 1. Define inputs for the selected flow rate ---
selected_index = 11  # Example index from your sweep
chosen_flow_slpm = undepressure_df.loc[selected_index, "Max flow rates (SLPM)"]
control_rate = undepressure_df.loc[selected_index, "Max DP rates (PSI/min)"]

# --- 2. Generate Valve Status History for Both Cases ---

# Lower Bound: Dynamic state simulation using your function
valve_status_opt, _ = run_underpressure_simulation(
    dt_array=dt_array,
    ambient_rate_array=dp_dt_array,
    initial_dp=setpoint_dp,
    valve_cracking_pressure=valve_cracking_pressure,
    valve_opening_rate=base_valve_rate,
    control_rate=control_rate,
)

# Upper Bound: Pure memoryless rate filter (overwhelmed entries)
valve_status_cons = dp_dt_array > control_rate


# --- 3. Helper Function to Extract Event Durations (in Seconds) ---
def get_event_durations_seconds(status_history, dt_array):
    labeled_array, num_features = label(status_history)
    durations_seconds = []
    for event_id in range(1, num_features + 1):
        event_mask = labeled_array == event_id
        total_dt_min = dt_array[event_mask].sum()
        durations_seconds.append(total_dt_min * 60.0)  # Convert to seconds
    return np.array(durations_seconds)


# Extract durations for both curves
durations_cons = get_event_durations_seconds(valve_status_cons, dt_array)
durations_opt = get_event_durations_seconds(valve_status_opt, dt_array)



In [114]:
# --- 4. Plot Overlaid Step Histograms in Plotly ---
fig_hist = go.Figure()

# Upper Bound Histogram (Step style: line only, no fill)
fig_hist.add_trace(
    go.Histogram(
        x=durations_cons,
        name=upper_bound_str,
        marker=dict(color="#75beff"),# line=dict(color="#d62728", width=2.5)),
        hovertemplate="<b>Upper Bound</b><br>Duration: %{x} s<br>Count: %{y}<extra></extra>",
        autobinx=False, # Tells Plotly not to override your settings
        xbins=dict(
            start=0,  # Where the first bin begins
            end=200,  # Where the last bin ends
            size=10,  # The width of each bin (e.g., 6 seconds)
        ),
    )
)

# Lower Bound Histogram (Step style: line only, no fill)
fig_hist.add_trace(
    go.Histogram(
        x=durations_opt,
        name=lower_bound_str,
        marker=dict(color="#007bff"),#, line=dict(color="#2ca02c", width=2.5)),
        hovertemplate="<b>Lower Bound</b><br>Duration: %{x} s<br>Count: %{y}<extra></extra>",
        autobinx=False,  # Tells Plotly not to override your settings
        xbins=dict(
            start=0,  # Where the first bin begins
            end=200,  # Where the last bin ends
            size=10,  # The width of each bin (e.g., 6 seconds)
        ),
    )
)

# Update layout for clean step overlaying
fig_hist.update_layout(
    title=dict(
        text=(
            f"<b>Check Valve Actuation Duration Distribution ({chosen_flow_slpm:.1f} SLPM)</b>"
            f"{assumptions_str}"
        ),
        font=dict(size=14),
        x=0.5,
        xanchor="center",
        y=0.93,
    ),
    xaxis=dict(
        title="Individual Event Duration (seconds)",
        gridcolor="rgba(200, 200, 200, 0.15)",
        # zeroline=False,
    ),
    yaxis=dict(
        title="Number of Occurrences (Event Count)",
        gridcolor="rgba(200, 200, 200, 0.15)",
        # zeroline=False,
    ),
    barmode="overlay",  # Essential for ensuring the steps sit on the same bins cleanly
    template="plotly_white",
    margin=dict(t=100, b=60, l=60, r=60),
    width=950,
    height=550,
    legend=dict(yanchor="top", y=0.95, xanchor="right", x=0.98),
)

fig_hist.write_html(f"event_durations_for_{chosen_flow_slpm:.0f}slpm_ctrl_rate.html")
fig_hist.show()